# Práctica clase 2. Bandidos multi-brazo: UCB y epsilon-greedy

**DIH DATAlife — Curso de Aprendizaje por Refuerzo desde cero**

En esta práctica vas a trabajar con el problema del bandido multi-brazo que viste en la clase de teoría (`clase02_bandidos.ipynb`). Vas a completar dos ejercicios:

1. **Implementar UCB (Upper Confidence Bound) desde cero**, completando la fórmula del bono de incertidumbre y la selección de acción
2. **Comparar el efecto de distintos valores de ε** en epsilon-greedy sobre el mismo bandido

Cada ejercicio tiene:

- Un enunciado con pistas
- Una celda de código con el andamiaje y comentarios `# TODO: ...` donde debes completar tú
- Una solución oculta, por si te atascas, que puedes desplegar haciendo clic

**Intenta resolver cada ejercicio por tu cuenta antes de mirar la solución.**


## Preparación: el bandido de partida

> Ejecuta estas celdas primero. Definen el entorno del bandido que vas a usar en los dos ejercicios, para que todos trabajéis sobre la misma base.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fijamos una semilla para que el experimento sea reproducible
np.random.seed(42)

n_brazos = 10
n_pasos = 1000


In [ ]:
class Bandido:
    """Bandido multi-brazo: cada brazo tiene un valor real fijo y desconocido
    para el agente. Al tirar de un brazo, se recibe una recompensa con ruido
    alrededor de ese valor real (como en el capítulo de teoría)."""

    def __init__(self, valores_reales):
        self.valores_reales = valores_reales
        self.n_brazos = len(valores_reales)

    def tirar_de_la_palanca(self, brazo):
        return np.random.normal(loc=self.valores_reales[brazo], scale=1.0)


# Creamos un bandido de ejemplo con valores reales aleatorios
valores_reales = np.random.normal(loc=0.0, scale=1.0, size=n_brazos)
bandido = Bandido(valores_reales)
print("Bandido creado con", bandido.n_brazos, "brazos")


---

## Ejercicio uno. Implementa UCB desde cero

Recuerda la idea de UCB de la clase de teoría: en vez de mirar solo tu estimación actual del valor de cada acción, le sumas un **bono de incertidumbre**. Ese bono es grande para los brazos poco probados y pequeño para los que ya conoces bien. En cada paso, eliges el brazo cuya suma de **valor estimado + bono** sea la más alta.

La fórmula del bono, para el brazo *a* en el paso *t*, es:

$$\text{bono}(a) = c \cdot \sqrt{\frac{\ln(t)}{N(a)}}$$

donde `N(a)` es el número de veces que has probado el brazo *a*, y `c` es una constante que controla cuánto pesa la exploración.

### Pistas

- El bono crece con `log(t)`: cuanto más avanza el tiempo, más "vale la pena" seguir revisando brazos poco probados
- El bono decrece con la raíz cuadrada de `N(a)`: cuantas más veces pruebas un brazo, menos incertidumbre te queda sobre su valor
- Usa `np.log(t)` y `np.sqrt(...)` de `numpy`
- Para elegir el brazo con la suma más alta, usa `np.argmax(valores_estimados + bono)`
- La parte de "probar cada brazo sin estrenar al menos una vez" ya está resuelta en el andamiaje: complétala solo si te la piden en el `# TODO`

**Completa los dos `# TODO` de la función `ejecutar_ucb`.**


In [ ]:
def ejecutar_ucb(bandido, c, n_pasos):
    """Estrategia UCB: combina el valor estimado con un bono de incertidumbre
    que favorece a los brazos poco probados."""
    valores_estimados = np.zeros(bandido.n_brazos)
    conteo_intentos = np.zeros(bandido.n_brazos)
    recompensas = np.zeros(n_pasos)

    for t in range(1, n_pasos + 1):
        # Si algún brazo aún no se ha probado, se elige primero (bono "infinito")
        brazos_sin_probar = np.where(conteo_intentos == 0)[0]
        if len(brazos_sin_probar) > 0:
            brazo = brazos_sin_probar[0]
        else:
            # TODO: calcula el bono de incertidumbre para cada brazo
            # bono(a) = c * sqrt(log(t) / N(a))
            bono = None

            # TODO: elige el brazo cuya suma "valor estimado + bono" sea la más alta
            brazo = None

        recompensa = bandido.tirar_de_la_palanca(brazo)
        conteo_intentos[brazo] += 1

        # Actualización incremental de la media muestral (igual que en la clase de teoría)
        paso = 1.0 / conteo_intentos[brazo]
        valores_estimados[brazo] += paso * (recompensa - valores_estimados[brazo])

        recompensas[t - 1] = recompensa

    return recompensas


# Ejecuta tu implementación con c = 2.0
recompensas_ucb = ejecutar_ucb(bandido, c=2.0, n_pasos=n_pasos)
recompensa_media_acumulada = np.cumsum(recompensas_ucb) / np.arange(1, n_pasos + 1)

plt.figure(figsize=(8, 4))
plt.plot(recompensa_media_acumulada)
plt.xlabel("Paso de tiempo")
plt.ylabel("Recompensa media acumulada")
plt.title("Tu implementación de UCB")
plt.grid(alpha=0.3)
plt.show()

print("Recompensa media acumulada final:", round(recompensa_media_acumulada[-1], 3))


<details>
<summary>💡 Ver solución</summary>

```python
def ejecutar_ucb(bandido, c, n_pasos):
    """Estrategia UCB: combina el valor estimado con un bono de incertidumbre
    que favorece a los brazos poco probados."""
    valores_estimados = np.zeros(bandido.n_brazos)
    conteo_intentos = np.zeros(bandido.n_brazos)
    recompensas = np.zeros(n_pasos)

    for t in range(1, n_pasos + 1):
        # Si algún brazo aún no se ha probado, se elige primero (bono "infinito")
        brazos_sin_probar = np.where(conteo_intentos == 0)[0]
        if len(brazos_sin_probar) > 0:
            brazo = brazos_sin_probar[0]
        else:
            # Bono de incertidumbre: crece con log(t), decrece con la raíz
            # cuadrada del número de veces que se ha probado cada brazo
            bono = c * np.sqrt(np.log(t) / conteo_intentos)

            # Elegimos el brazo con mayor "valor estimado + bono"
            brazo = np.argmax(valores_estimados + bono)

        recompensa = bandido.tirar_de_la_palanca(brazo)
        conteo_intentos[brazo] += 1

        paso = 1.0 / conteo_intentos[brazo]
        valores_estimados[brazo] += paso * (recompensa - valores_estimados[brazo])

        recompensas[t - 1] = recompensa

    return recompensas


recompensas_ucb = ejecutar_ucb(bandido, c=2.0, n_pasos=n_pasos)
recompensa_media_acumulada = np.cumsum(recompensas_ucb) / np.arange(1, n_pasos + 1)
print("Recompensa media acumulada final:", round(recompensa_media_acumulada[-1], 3))
```
</details>


---

## Ejercicio dos. Compara distintos valores de epsilon

En la clase de teoría viste que ε controla cuánto explora epsilon-greedy: un ε pequeño explota casi siempre, y un ε grande explora mucho más. Ahora vas a comprobarlo tú misma o tú mismo.

Vas a entrenar epsilon-greedy tres veces sobre el mismo bandido, una por cada valor de la lista `[0.01, 0.1, 0.3]`, y a dibujar las tres curvas de recompensa media acumulada en una misma gráfica para compararlas.

### Pistas

- La función `ejecutar_epsilon_greedy` ya está completa: no necesitas tocarla
- Para cada `epsilon` de la lista, repite el experimento `n_repeticiones` veces y promedia, igual que en la clase de teoría, para suavizar el azar
- No olvides regenerar `valores_reales` en cada repetición (así el promedio no depende de una única configuración de brazos)
- Usa `plt.plot(curva, label=f"epsilon = {epsilon}")` dentro del bucle para añadir cada curva a la misma gráfica
- Al final, no olvides `plt.legend()` para distinguir las tres curvas

**Completa los `# TODO` del bucle.**


In [ ]:
def ejecutar_epsilon_greedy(bandido, epsilon, n_pasos):
    """Estrategia epsilon-greedy: explora al azar con probabilidad epsilon,
    y el resto de las veces elige la acción con mejor valor estimado."""
    valores_estimados = np.zeros(bandido.n_brazos)
    conteo_intentos = np.zeros(bandido.n_brazos)
    recompensas = np.zeros(n_pasos)

    for t in range(n_pasos):
        if np.random.random() < epsilon:
            brazo = np.random.randint(bandido.n_brazos)
        else:
            brazo = np.argmax(valores_estimados)

        recompensa = bandido.tirar_de_la_palanca(brazo)
        conteo_intentos[brazo] += 1
        paso = 1.0 / conteo_intentos[brazo]
        valores_estimados[brazo] += paso * (recompensa - valores_estimados[brazo])
        recompensas[t] = recompensa

    return recompensas


valores_epsilon = [0.01, 0.1, 0.3]
n_repeticiones = 100

plt.figure(figsize=(10, 5))

# TODO: recorre "valores_epsilon" con un bucle for
for epsilon in valores_epsilon:
    recompensas_totales = np.zeros(n_pasos)

    # TODO: repite el experimento "n_repeticiones" veces y suma los resultados
    for _ in range(n_repeticiones):
        # TODO: regenera "valores_reales" al azar en cada repetición
        valores_reales_rep = None

        # TODO: crea un Bandido con esos valores reales
        bandido_rep = None

        # TODO: acumula las recompensas de ejecutar_epsilon_greedy en recompensas_totales
        pass

    recompensas_totales /= n_repeticiones

    # TODO: calcula la recompensa media acumulada a partir de "recompensas_totales"
    curva = None

    plt.plot(curva, label=f"epsilon = {epsilon}")

plt.xlabel("Paso de tiempo")
plt.ylabel("Recompensa media acumulada")
plt.title("Comparación de valores de epsilon en epsilon-greedy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


<details>
<summary>💡 Ver solución</summary>

```python
valores_epsilon = [0.01, 0.1, 0.3]
n_repeticiones = 100

plt.figure(figsize=(10, 5))

for epsilon in valores_epsilon:
    recompensas_totales = np.zeros(n_pasos)

    for _ in range(n_repeticiones):
        # Regeneramos los valores reales en cada repetición para que el
        # promedio no dependa de una única configuración de brazos
        valores_reales_rep = np.random.normal(loc=0.0, scale=1.0, size=n_brazos)
        bandido_rep = Bandido(valores_reales_rep)
        recompensas_totales += ejecutar_epsilon_greedy(bandido_rep, epsilon, n_pasos)

    recompensas_totales /= n_repeticiones
    curva = np.cumsum(recompensas_totales) / np.arange(1, n_pasos + 1)
    plt.plot(curva, label=f"epsilon = {epsilon}")

plt.xlabel("Paso de tiempo")
plt.ylabel("Recompensa media acumulada")
plt.title("Comparación de valores de epsilon en epsilon-greedy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
```

**Qué deberías observar:** con ε = 0,01 casi no se explora, así que el aprendizaje es lento al principio. Con ε = 0,3 se explora tanto que la curva se queda más baja incluso a largo plazo, porque se siguen desperdiciando muchos intentos en brazos malos. El valor intermedio, ε = 0,1, suele encontrar el mejor equilibrio en este experimento.
</details>


---

## Para seguir practicando

Si te ha sobrado tiempo, retoma las ideas de "Ponlo en práctica" del manual:

- **Cambia el número de brazos** `n_brazos` (por ejemplo, a 5 o a 20) y comprueba si UCB mantiene su ventaja sobre epsilon-greedy
- **Prueba distintos valores de `c`** en tu UCB y anota a partir de qué valor la estrategia empieza a explorar demasiado

**¡Comparte con tu equipo qué combinación de parámetros encontró antes el mejor brazo!**
